In [ ]:
from pathlib import Path

import h5py
import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from skimage.graph import route_through_array
from matplotlib.collections import LineCollection
from pathlib import Path
import copy
from scipy.ndimage import convolve
from skimage.morphology import binary_dilation
from skimage.graph import route_through_array

DATA_DIR = Path("/Users/sylvi/topo_data/temp/")
IMAGE_FILE = "20251209_TAF_pICOz_MQ_50_percent_damage.0_00010.topostats"
FILE_PATH = DATA_DIR / IMAGE_FILE
GRAIN_IDX = "1"

height_threshold_nm = 2.2  # Minimum height in nm required for a crossing to be considered a true crossing.
branch_explore_distance_nm = 2.5  # Maximum distance in nm to explore along a branch from a node for creating
cost_image_exponent = 4.0  # Exponent to apply to the cost image to increase the cost of using lower pixels
cost_image_base = 0.01  # base cost to prevent zero cost pixels, just in case
crossing_correction_strand_minimum_height_nm = 1.0  # minimum height for a strand to be used for crossing correction

crossing_data = []


def get_neighbouring_true_pixels(
    mask: npt.NDArray[np.bool_], coord: npt.NDArray[np.integer]
) -> list[npt.NDArray[np.integer]]:
    """
    Get the coordinates of neighbouring true pixels in a binary mask.

    Parameters
    ----------
    mask : npt.NDArray[np.bool_]
        The binary mask.
    coord : npt.NDArray[np.integer]
        The coordinate to get neighbours for.

    Returns
    -------
    list[npt.NDArray[np.integer]]
        A list of coordinates of neighbouring true pixels.
    """
    neighbours = []
    for dx in [-1, 0, 1]:
        for dy in [-1, 0, 1]:
            if dx == 0 and dy == 0:
                continue
            neighbour_coord = coord + np.array([dx, dy])
            if (
                neighbour_coord[0] >= 0
                and neighbour_coord[0] < mask.shape[0]
                and neighbour_coord[1] >= 0
                and neighbour_coord[1] < mask.shape[1]
            ):
                if mask[neighbour_coord[0], neighbour_coord[1]]:
                    neighbours.append(neighbour_coord)
    return neighbours


def get_point_along_branch(
    skeleton: npt.NDArray[np.bool_], start_coord: npt.NDArray[np.integer], distance_px: float
) -> tuple[npt.NDArray[np.integer], npt.NDArray[np.integer]]:
    """
    Get the coordinate and path taken of a point along a branch at a specified distance from a starting coordinate.

    Parameters
    ----------
    skeleton : npt.NDArray[np.bool_]
        The skeleton image.
    start_coord : npt.NDArray[np.integer]
        The starting coordinate.
    distance_px : float
        The distance in pixels.

    Returns
    -------
    tuple[npt.NDArray[np.integer], npt.NDArray[np.integer]]
        The coordinate of the point along the branch at the specified distance, and the path taken to get there.
    """
    skeleton_tracker = copy.deepcopy(skeleton)
    distance_travelled_px = 0.0
    path_taken = [start_coord]
    current_coord = start_coord
    skeleton_tracker[current_coord[0], current_coord[1]] = False
    neighbours = get_neighbouring_true_pixels(skeleton_tracker, current_coord)
    # note: if we ever encounter more than one neighbour, we stop and return that point, as it's
    # a crossing / node, and we don't want to have to define how to deicde where to go from there.
    while distance_travelled_px < distance_px:
        if len(neighbours) == 0:
            # we've reached the end of the branch before reaching the desired distance, so we return the last point
            break
        elif len(neighbours) > 1:
            # we've reached a crossing / node before reaching the desired distance, so we return the current point
            break
        else:
            next_coord = neighbours[0]
            distance_travelled_px += np.linalg.norm(next_coord - current_coord)
            current_coord = next_coord
            path_taken.append(current_coord)
            skeleton_tracker[current_coord[0], current_coord[1]] = False
            neighbours = get_neighbouring_true_pixels(skeleton_tracker, current_coord)
    return current_coord, np.array(path_taken)


with h5py.File(FILE_PATH, "r") as f:
    image = f["image"][:]
    pixel_to_nm_scaling = f["pixel_to_nm_scaling"][()]
    print(f"pixel to nm scaling: {pixel_to_nm_scaling:.2f} nm/px")
    plt.imshow(image, cmap="afmhot", origin="lower")
    plt.show()
    print(f.keys())
    graincrops = f["grain_crops"]

    for grain_index, graincrop in graincrops.items():
        print(f"\n\n\n\n\ngrain index: {grain_index}")
        grain_image = graincrop["image"][:]
        grain_image_normalised = (grain_image - np.min(grain_image)) / (
            np.max(grain_image) - np.min(grain_image) + 1e-8
        )
        grain_cost_image = (1.0 - grain_image_normalised) ** cost_image_exponent + cost_image_base
        plt.imshow(grain_cost_image, cmap="gray", origin="lower")
        plt.title(f"Grain {grain_index} cost image")
        plt.show()
        bbox = graincrop["bbox"]
        # print(graincrop.keys())
        disordered_trace = graincrop["disordered_trace"]
        # print(disordered_trace.keys())
        disordered_trace_images = disordered_trace["images"]
        # print(disordered_trace_images.keys())
        branch_indexes_image = disordered_trace_images["branch_indexes"][:]
        branch_indexes_nonzero = np.argwhere(branch_indexes_image > 0)
        # print(branch_indexes_nonzero)
        pruned_skeleton = disordered_trace_images["pruned_skeleton"][:].astype(np.int32)

        result_corrected_pruned_skeleton = copy.deepcopy(pruned_skeleton)

        # find the nodes
        convolved_skeleton = convolve(pruned_skeleton, np.ones((3, 3)))
        convolved_skeleton[pruned_skeleton == 0] = 0

        print(convolved_skeleton.shape)
        nodes = graincrop["nodes"]
        print(f"nodes: {nodes.keys()}")

        for node_index, node in nodes.items():
            print(f"\n\n\nnode index: {node_index}")
            print(node.keys())

            node_area_skeleton = node["node_area_skeleton"][:]
            print(np.unique(node_area_skeleton))
            fig, ax = plt.subplots(1, 1, figsize=(10, 10))
            ax.imshow(node_area_skeleton, cmap="gray", origin="lower")
            plt.title(f"Node {node_index} area skeleton")
            plt.show()

            mask_node = np.where(node_area_skeleton == 3, 1, 0)
            mask_branches = np.where(node_area_skeleton == 1, 1, 0)

            plt.imshow(mask_branches, cmap="gray", origin="lower")
            plt.title(f"Node {node_index} branch mask")
            plt.show()

            # get the maximum height of any node pixel from the image
            node_pixel_coords = np.argwhere(mask_node == 1)
            node_pixel_heights = grain_image[node_pixel_coords[:, 0], node_pixel_coords[:, 1]]
            max_node_pixel_height = np.max(node_pixel_heights)
            if max_node_pixel_height > height_threshold_nm:
                # This is a true crossing
                print(f"node {node_index} is a true crossing with max height {max_node_pixel_height:.2f} nm, skipping")
                crossing_data.append(
                    {
                        "grain_index": grain_index,
                        "node_index": node_index,
                        "true_crossing": True,
                        "max_node_pixel_height": max_node_pixel_height,
                        "coords": node_pixel_coords,
                        "centroid": np.mean(node_pixel_coords, axis=0),
                        "bbox": bbox,
                    }
                )

                # do true crossing adjustments here
                # TODO: true crossing adjustments
            else:
                crossing_data.append(
                    {
                        "grain_index": grain_index,
                        "node_index": node_index,
                        "true_crossing": False,
                        "max_node_pixel_height": max_node_pixel_height,
                        "coords": node_pixel_coords,
                        "centroid": np.mean(node_pixel_coords, axis=0),
                        "bbox": bbox,
                    }
                )

                # false crossing fixing
                mask_dilated_node = binary_dilation(mask_node, footprint=np.ones((3, 3)))
                mask_dilated_node_branch_intersections = mask_dilated_node * mask_branches
                fig, ax = plt.subplots(1, 1, figsize=(10, 10))
                ax.imshow(mask_dilated_node_branch_intersections, cmap="gray", origin="lower")
                plt.title(f"Node {node_index} dilated node-branch intersections")
                plt.show()

                branch_starts = np.argwhere(mask_dilated_node_branch_intersections == 1)
                if branch_starts.shape[0] != 4:
                    print(
                        f"Warning: expected 4 branch starts for node {node_index} but found {branch_starts.shape[0]}, skipping"
                    )
                    continue

                print(f"correcting node {node_index}, branch starts: {branch_starts}")

                fig, ax = plt.subplots(1, 1, figsize=(10, 10))
                ax.imshow(pruned_skeleton, cmap="gray", origin="lower")

                point_a, path_a = get_point_along_branch(
                    skeleton=mask_branches,
                    start_coord=branch_starts[0],
                    distance_px=branch_explore_distance_nm / pixel_to_nm_scaling,
                )
                result_corrected_pruned_skeleton[path_a[:, 0], path_a[:, 1]] = 0
                point_b, path_b = get_point_along_branch(
                    skeleton=mask_branches,
                    start_coord=branch_starts[1],
                    distance_px=branch_explore_distance_nm / pixel_to_nm_scaling,
                )
                result_corrected_pruned_skeleton[path_b[:, 0], path_b[:, 1]] = 0
                point_c, path_c = get_point_along_branch(
                    skeleton=mask_branches,
                    start_coord=branch_starts[2],
                    distance_px=branch_explore_distance_nm / pixel_to_nm_scaling,
                )
                result_corrected_pruned_skeleton[path_c[:, 0], path_c[:, 1]] = 0
                point_d, path_d = get_point_along_branch(
                    skeleton=mask_branches,
                    start_coord=branch_starts[3],
                    distance_px=branch_explore_distance_nm / pixel_to_nm_scaling,
                )
                result_corrected_pruned_skeleton[path_d[:, 0], path_d[:, 1]] = 0

                # also remove the node coords from the result skeleton
                result_corrected_pruned_skeleton[mask_node == 1] = 0

                ax.scatter(
                    [point_a[1], point_b[1], point_c[1], point_d[1]],
                    [point_a[0], point_b[0], point_c[0], point_d[0]],
                    color="blue",
                    s=100,
                )

                possible_combinations = [
                    ((point_a, point_b), (point_c, point_d)),
                    ((point_a, point_c), (point_b, point_d)),
                    ((point_a, point_d), (point_b, point_c)),
                ]

                best_combination: (
                    tuple[tuple[np.ndarray, np.ndarray, np.ndarray], tuple[np.ndarray, np.ndarray, np.ndarray]] | None
                ) = None
                highest_lowest_height_along_paths = -np.inf
                for combination in possible_combinations:
                    pair_1, pair_2 = combination
                    # get the costs for pathfinding between the pairs
                    start_1 = pair_1[0]
                    end_1 = pair_1[1]
                    start_2 = pair_2[0]
                    end_2 = pair_2[1]
                    path_1_coords, _path_1_cost = route_through_array(
                        grain_cost_image, start_1, end_1, fully_connected=True
                    )
                    path_1_coords = np.array(path_1_coords)
                    path_1_heights = grain_image[path_1_coords[:, 0], path_1_coords[:, 1]]
                    path_2_coords, _path_2_cost = route_through_array(
                        grain_cost_image, start_2, end_2, fully_connected=True
                    )
                    path_2_coords = np.array(path_2_coords)
                    path_2_heights = grain_image[path_2_coords[:, 0], path_2_coords[:, 1]]

                    lowest_height_along_paths = np.min(np.concatenate([path_1_heights, path_2_heights]))
                    if lowest_height_along_paths > highest_lowest_height_along_paths:
                        highest_lowest_height_along_paths = lowest_height_along_paths
                        best_combination = ((start_1, end_1, path_1_coords), (start_2, end_2, path_2_coords))
                if best_combination is None:
                    raise ValueError(f"this should never happen, debug this.")

                ax.plot(best_combination[0][2][:, 1], best_combination[0][2][:, 0], color="cyan", linewidth=2)
                ax.plot(best_combination[1][2][:, 1], best_combination[1][2][:, 0], color="cyan", linewidth=2)
                if highest_lowest_height_along_paths < crossing_correction_strand_minimum_height_nm:
                    print(
                        f"node {node_index} cannot be corrected, highest lowest height along paths is {highest_lowest_height_along_paths:.2f} nm, skipping"
                    )
                    continue
                else:
                    print(f"found correction for node")
                    # apply the correction to the pruned skeleton
                    for _, _, path_coords in best_combination:
                        for coord in path_coords:
                            result_corrected_pruned_skeleton[coord[0], coord[1]] = 1
                plt.show()

    print("*" * 200)
    print("*" * 200)
    print("*" * 200)

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(image, cmap="afmhot", origin="lower")
    for crossing in crossing_data:
        bbox = crossing["bbox"]
        if crossing["true_crossing"]:
            ax.scatter(
                crossing["centroid"][1] + bbox[1],
                crossing["centroid"][0] + bbox[0],
                color="green",
                edgecolors="white",
                s=100,
            )
        else:
            ax.scatter(
                crossing["centroid"][1] + bbox[1],
                crossing["centroid"][0] + bbox[0],
                color="red",
                edgecolors="white",
                s=100,
            )
    plt.title("True crossings identified in the image")
    plt.show()

    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    ax.imshow(result_corrected_pruned_skeleton, cmap="gray", origin="lower")
    plt.title("Corrected pruned skeleton")
    plt.show()